In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込みエラー: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 24. デコーディング戦略

> **学習目標**
> - Greedy, Beam Search, Top-k, Top-p, Temperature
> - ·
> - Speculative Decoding

## 24.1 問題

LLM 分布 $P(w_t | w_{<t})$ 出力.
 = .

: ** vs **

## 24.2 Greedy Decoding

 :
$$w_t = \arg\max_w P(w | w_{<t})$$

-
- ,


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

#    (10 ,  20)
torch.manual_seed(0)
vocab_size = 20
n_steps = 10
logits_seq = torch.randn(n_steps, vocab_size)

def softmax(x, tau=1.0):
    x = x / tau
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

def greedy_decode(logits_seq):
    """Greedy:  Step argmax."""
    tokens = []
    for logits in logits_seq:
        tokens.append(logits.argmax().item())
    return tokens

greedy_tokens = greedy_decode(logits_seq)
print(f"Greedy: {greedy_tokens}")


## 24.3 Temperature Sampling

$$P_\tau(w) = \frac{\exp(\log P(w) / \tau)}{\sum_{w'} \exp(\log P(w') / \tau)}$$

- $\tau \to 0$: Greedy ( 分布)
- $\tau = 1$:
- $\tau \to \infty$: 分布 ( )


In [ ]:
# Temperature 
logits = torch.randn(20)  # 20   
print(f": {logits.numpy().round(2)}")
print()
for tau in [0.3, 1.0, 2.0, 5.0]:
    p = softmax(logits.numpy(), tau=tau)
    print(f"tau={tau}: {p.round(4)}  entropy={-np.sum(p*np.log(p+1e-12)):.3f}")
print("\n=> tau   Distribution(),  ().")


## 24.4 Top-k Sampling

 $k$ , 0:
$$\mathcal{V}_k = \text{top-}k\{P(w)\}$$
$$\hat{P}(w) = \frac{P(w)}{\sum_{w' \in \mathcal{V}_k} P(w')} \text{ for } w \in \mathcal{V}_k, \text{ else } 0$$

## 24.5 Top-p (Nucleus) Sampling

 $p$ :
$$\mathcal{V}_p = \min \{S : \sum_{w \in S} P(w) \geq p\}$$

: 分布 , .


In [ ]:
# Top-k, Top-p 
def top_k_sampling(logits, k=5, tau=1.0):
    """Top-k Sample."""
    p = softmax(logits.numpy(), tau=tau)
    #  k 
    top_idx = np.argsort(p)[-k:]
    mask = np.zeros_like(p)
    mask[top_idx] = p[top_idx]
    # Normalization
    mask = mask / mask.sum()
    return np.random.choice(len(p), p=mask)

def top_p_sampling(logits, p=0.9, tau=1.0):
    """Top-p (Nucleus) Sample."""
    probs = softmax(logits.numpy(), tau=tau)
    #   
    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    # 
    cumsum = np.cumsum(sorted_probs)
    # p    
    cutoff = np.searchsorted(cumsum, p) + 1
    #  cutoff  
    keep = sorted_idx[:cutoff]
    new_p = np.zeros_like(probs)
    new_p[keep] = probs[keep]
    new_p = new_p / new_p.sum()
    return np.random.choice(len(probs), p=new_p)

# 可視化:     
np.random.seed(42)
logits = np.random.randn(20) * 2
probs = softmax(logits, tau=1.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Top-k  計算
top5_idx = np.argsort(probs)[-5:]
mask_topk = np.zeros(20)
mask_topk[top5_idx] = probs[top5_idx]
mask_topk = mask_topk / mask_topk.sum()

# Top-p  計算
sorted_idx = np.argsort(probs)[::-1]
sorted_probs = probs[sorted_idx]
cumsum = np.cumsum(sorted_probs)
cutoff = np.searchsorted(cumsum, 0.9) + 1
keep = sorted_idx[:cutoff]
mask_p = np.zeros(20); mask_p[keep] = probs[keep]
mask_p = mask_p / mask_p.sum()

# temperature
mask_t = softmax(logits, tau=0.5)

strategies = [
    ('Greedy', np.eye(20)[np.argmax(probs)]),
    ('Top-k=5', mask_topk),
    ('Top-p=0.9', mask_p),
    ('Temperature=0.5', mask_t),
]

for ax, (name, mask) in zip(axes, strategies):
    ax.bar(range(20), probs, alpha=0.3, color='gray', label='Original Distribution')
    ax.bar(range(20), mask, alpha=0.7, color='red', label='Line ')
    ax.set_title(name)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../figures/ch24_decoding_strategies.png', dpi=100, bbox_inches='tight')
plt.show()


## 24.6 Beam Search

$B$ (beam) , :

: $\sum_t \log P(w_t | w_{<t})$

- $B$ = 4~10
-
- LLM (, )


In [ ]:
# Beam Search  
def beam_search(initial_tokens, get_logits_fn, beam_width=4, max_len=20):
    """ beam search."""
    beams = [(initial_tokens, 0.0)]  # (tokens, log_prob)
    for step in range(max_len):
        all_candidates = []
        for tokens, score in beams:
            logits = get_logits_fn(tokens)
            log_probs = F.log_softmax(logits[-1], dim=-1)
            #  beam_width
            topk = log_probs.topk(beam_width)
            for i in range(beam_width):
                new_tokens = tokens + [topk.indices[i].item()]
                new_score = score + topk.values[i].item()
                all_candidates.append((new_tokens, new_score))
        #  beam_width Line
        all_candidates.sort(key=lambda x: x[1], reverse=True)
        beams = all_candidates[:beam_width]
    return beams[0]  #  Scores

#   Function
def fake_logits_fn(tokens):
    return torch.randn(len(tokens), 50)  #  50

result_tokens, result_score = beam_search([1, 2, 3], fake_logits_fn, beam_width=4, max_len=10)
print(f"Beam Search 結果: tokens={result_tokens}, score={result_score:.4f}")


## 24.7 Speculative Decoding

****: モデル(draft) $k$ , モデル(target) 検証.

1. Draft モデル: $w_1^d, \ldots, w_k^d$ 
2. Target モデル: $P_T(w_1), \ldots, P_T(w_k)$ 計算 ( 順伝播)
3. ,

: 2-3x 高速化 (Medusa, EAGLE )


## 24.8 Repetition Penalty

 : .

$$\logit'(w) = \begin{cases} \logit(w) / p & \text{if } w \in \text{generated} \\ \logit(w) & \text{otherwise} \end{cases}$$

$p > 1$ ( 1.1~1.3).

## 24.9 要点

| | | |
|---|---|---|
| Greedy | $\arg\max P$ | , |
| Temperature | $\sigma(z/\tau)$ | 分布 |
| Top-k | $\text{top-}k$ | k |
| Top-p | $p$ | |
| Beam | $B$ | ↑, ↓ |
| Speculative | draft + target | 2-3x |

## 演習問題

1. Greedy, Top-k=5, Top-p=0.9, Temperature=0.7 比較.
2. Temperature 0.3, 0.7, 1.0, 1.5 比較.
3. Beam Search 1, 4, 8 比較.
4. Repetition Penalty 1.0, 1.1, 1.3 .
5. Speculative Decoding 2-3x .

> 解答: `solutions/ch24_solutions.ipynb`
